<a href="https://colab.research.google.com/github/Anthonyi12/AnthonyPortfolio/blob/main/Delaney_ESOL_LR/LR_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
URL = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
df = pd.read_csv(URL)
df.to_csv("delaney-processed.csv", index=False)

In [ ]:
df

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.770,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.300,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.060,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.870,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.330,c1ccsc1
...,...,...,...,...,...,...,...,...,...,...
1123,halothane,-2.608,1,197.381,0,0,0,0.00,-1.710,FC(F)(F)C(Cl)Br
1124,Oxamyl,-0.908,1,219.266,1,0,1,71.00,0.106,CNC(=O)ON=C(SC)C(=O)N(C)C
1125,Thiometon,-3.323,1,246.359,0,0,7,18.46,-3.091,CCSCCSP(=S)(OC)OC
1126,2-Methylbutane,-2.245,1,72.151,0,0,1,0.00,-3.180,CCC(C)C


In [ ]:
print(df.shape)
print(df.columns.tolist())

(1128, 10)
['Compound ID', 'ESOL predicted log solubility in mols per litre', 'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors', 'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area', 'measured log solubility in mols per litre', 'smiles']


In [ ]:
from rdkit import Chem

TARGET = "measured log solubility in mols per litre"

# 1. remove spaces from SMILES
before = df["smiles"].copy()
df["smiles"] = df["smiles"].str.strip()
print(f"SMILES with Stray space: {(before != df["smiles"]).sum()}")

# 2. Turn each SMILES into an RDKit molecule
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
print(f"Unreadable Smiles: {df["mol"].isna().sum()}")

# 3. Find duplicates using the standard (canonical) SMILES
df["canon"] = df["mol"].apply(Chem.MolToSmiles)
# every row that's a duplicate has a copy flagged.
print(f"Rows sharing a structure: {df["canon"].duplicated(keep=False).sum()}")

# 4. Merge duplicates (average their solubility), keep only what's needed
df = df.groupby("canon", as_index=False).agg(
    {TARGET: "mean", "mol": "first", "Compound ID": "first"}
)

print(f"Molecules after cleaning: {len(df)}")

SMILES with Stray space: 212
Unreadable Smiles: 0
Rows sharing a structure: 22
Molecules after cleaning: 1117


In [ ]:
df

,canon,measured log solubility in mols per litre,mol,Compound ID
0,BrC(Br)(Br)Br,-3.14,<rdkit.Chem.rdchem.Mol object at 0x7f33bc5b6b90>,Tetrabromomethane
1,BrC(Br)Br,-1.91,<rdkit.Chem.rdchem.Mol object at 0x7f33bc5a42e0>,Tribromomethane
2,BrCBr,-1.17,<rdkit.Chem.rdchem.Mol object at 0x7f33bc5a7bc0>,Dibromomethane
3,BrCCBr,-1.68,<rdkit.Chem.rdchem.Mol object at 0x7f33bc5c6b90>,"1,2-Dibromoethane"
4,Brc1cc(Br)c(Br)cc1Br,-6.98,<rdkit.Chem.rdchem.Mol object at 0x7f33bc57e340>,"1,2,4,5-Tetrabromobenzene"
...,...,...,...,...
1112,c1ccoc1,-0.82,<rdkit.Chem.rdchem.Mol object at 0x7f33bc5b6490>,Furane
1113,c1ccsc1,-1.33,<rdkit.Chem.rdchem.Mol object at 0x7f33bc94aea0>,Thiophene
1114,c1cnc2c(c1)ccc1ncccc12,-2.68,<rdkit.Chem.rdchem.Mol object at 0x7f33bc57f6f0>,"1,7-phenantroline"
1115,c1cnc2ncncc2n1,0.02,<rdkit.Chem.rdchem.Mol object at 0x7f33bc5c4890>,Pteridine


In [ ]:
from rdkit.Chem import Descriptors, Crippen
# same features Delaney used for his LR-Model
def features(mol):
  aromatic = sum(a.GetIsAromatic() for a in mol.GetAtoms())
  return {
      "LogP": Crippen.MolLogP(mol),
      "MW":   Descriptors.MolWt(mol),
      "RB":   Descriptors.NumRotatableBonds(mol),
      "AP":   aromatic / mol.GetNumHeavyAtoms(),
  }

X = pd.DataFrame([features(m) for m in df["mol"]])
y = df[TARGET].values

print(X.shape)
print(X.describe().round(4))

(1117, 4)
            LogP         MW         RB         AP
count  1117.0000  1117.0000  1117.0000  1117.0000
mean      2.4534   203.7074     2.1737     0.3642
std       1.8451   102.9441     2.6434     0.3440
min      -7.5714    16.0430     0.0000     0.0000
25%       1.4176   120.6230     0.0000     0.0000
50%       2.3392   182.1720     1.0000     0.3750
75%       3.3918   270.3720     3.0000     0.6667
max      10.3886   780.9490    23.0000     1.0000


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)
print("Train: ", X_train.shape, "Test: ", X_test.shape)

Train:  (893, 4) Test:  (224, 4)


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def report(name, y_true, y_pred):
  rmse = mean_squared_error(y_true, y_pred)**0.5
  mae = mean_absolute_error(y_true, y_pred)
  r2 = r2_score(y_true, y_pred)
  print(f"{name:20s} RMSE {rmse:.3f} MAE {mae:.3f} R2 {r2:.3f}")

# Baseline - always predicts average logS
baseline = DummyRegressor().fit(X_train, y_train)
report("Baseline (mean)", y_test, baseline.predict(X_test))

# Linear Regression with features scaled
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)
report("Linear Reg", y_test, model.predict(X_test))

Baseline (mean)      RMSE 2.040 MAE 1.641 R2 -0.000
Linear Reg           RMSE 1.025 MAE 0.775 R2 0.748


In [ ]:
y_pred = model.predict(X_test)
y_pred_scaled_df = pd.DataFrame([f"{10**num:e}" for num in y_pred]).rename(columns={0: "Scaled Model Predicitions"})
y_pred_scaled_df

,Scaled Model Predicitions
0,1.559254e-01
1,4.959248e-06
2,6.984975e-03
3,7.121073e-02
4,1.360689e-04
...,...
219,1.364017e-05
220,1.333613e-05
221,2.209728e-07
222,1.582619e-01


In [ ]:
y_test_scaled_df = pd.DataFrame([f"{10**(num):e}" for num in y_test]).rename(columns={0: "Scaled Test Values"})
y_test_scaled_df

,Scaled Test Values
0,5.058247e-03
1,3.801894e-05
2,8.241381e-03
3,5.128614e-01
4,1.096478e-04
...,...
219,1.000000e-06
220,2.089296e-07
221,1.380384e-07
222,2.398833e-01


Below is the comparison between the Scaled MOdel Predictions and the Scaled Test Values.

In [ ]:
y_pred_scaled_df.merge(y_test_scaled_df, right_index=True, left_index=True)

,Scaled Model Predicitions,Scaled Test Values
0,1.559254e-01,5.058247e-03
1,4.959248e-06,3.801894e-05
2,6.984975e-03,8.241381e-03
3,7.121073e-02,5.128614e-01
4,1.360689e-04,1.096478e-04
...,...,...
219,1.364017e-05,1.000000e-06
220,1.333613e-05,2.089296e-07
221,2.209728e-07,1.380384e-07
222,1.582619e-01,2.398833e-01


Reproduces the results from the Delaney research:
https://pubs.acs.org/jcisd8/article-abstract/44/3/1000/3504546/ESOL-Estimating-Aqueous-Solubility-Directly-from?redirectedFrom=fulltext

Their findings where within a factor of 5-8 ($10^{0.775} \approx 5.956$).